# Task: Linear vs Polynomial Regression

## Objective
Build and compare Linear Regression and Polynomial Regression models then select the best performing model and interpret its learned equation.


## Dataset
Use the **Boston Housing dataset**.
    
    from sklearn.datasets import load_boston

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import make_pipeline

try:
    from sklearn.datasets import load_boston
    boston = load_boston()
    X = pd.DataFrame(boston.data, columns=boston.feature_names)
    y = pd.Series(boston.target, name="MEDV")
except Exception:
    from sklearn.datasets import fetch_openml
    boston = fetch_openml(name="boston", version=1, as_frame=True)
    X = boston.data.copy()
    for col in X.select_dtypes(include=["category"]).columns:
        X[col] = X[col].astype(float)
    y = pd.to_numeric(boston.target, errors="coerce").astype(float).rename("MEDV")

In [2]:
df = X.copy()
df["MEDV"] = y
df.head()

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.00632,18.0,2.31,0.0,0.538,6.575,65.2,4.0900,1.0,296.0,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0.0,0.469,6.421,78.9,4.9671,2.0,242.0,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0.0,0.469,7.185,61.1,4.9671,2.0,242.0,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0.0,0.458,6.998,45.8,6.0622,3.0,222.0,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0.0,0.458,7.147,54.2,6.0622,3.0,222.0,18.7,396.90,5.33,36.2


In [3]:
print("Dataset shape:", X.shape)
print("Target shape:", y.shape)
print("Features:", list(X.columns))

Dataset shape: (506, 13)
Target shape: (506,)
Features: ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO', 'B', 'LSTAT']


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=101
)
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (354, 13)
Test shape: (152, 13)


### 1. Data Preparation

In [5]:
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [6]:
linear_predictions = linear_model.predict(X_test)

In [7]:
linear_mae = mean_absolute_error(y_test, linear_predictions)
linear_mse = mean_squared_error(y_test, linear_predictions)
linear_rmse = np.sqrt(linear_mse)

print(f"Linear Regression MAE: {linear_mae:.3f}")
print(f"Linear Regression MSE: {linear_mse:.3f}")
print(f"Linear Regression RMSE: {linear_rmse:.3f}")

Linear Regression MAE: 3.836
Linear Regression MSE: 28.548
Linear Regression RMSE: 5.343


In [8]:
coefficients = pd.Series(linear_model.coef_, index=X.columns)
print("Linear model coefficients sorted by magnitude:")
print(coefficients.abs().sort_values(ascending=False))
print(f"Intercept: {linear_model.intercept_:.3f}")

Linear model coefficients sorted by magnitude:
NOX        17.748371
CHAS        3.754271
RM          3.247765
DIS         1.409161
PTRATIO     0.951781
LSTAT       0.597133
RAD         0.263881
CRIM        0.088505
ZN          0.050293
INDUS       0.020348
AGE         0.012001
TAX         0.010344
B           0.006116
dtype: float64
Intercept: 40.219


In [9]:
print("Linear regression training complete.")

Linear regression training complete.


### 2. Linear Regression Model
- Train a Linear Regression model
- Make predictions on the test set
- Compute evaluation metrics:
  - MAE
  - MSE
  - RMSE

In [10]:
poly_model = make_pipeline(
    PolynomialFeatures(degree=2, include_bias=False),
    LinearRegression()
)
poly_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('polynomialfeatures', ...), ('linearregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"degree degree: int or tuple (min_degree, max_degree), default=2If a single int is given, it specifies the maximal degree of thepolynomial features. If a tuple `(min_degree, max_degree)` is passed,then `min_degree` is the minimum and `max_degree` is the maximumpolynomial degree of the generated features. Note that `min_degree=0`and `min_degree=1` are equivalent as outputting the degree zero term isdetermined by `include_bias`.",2
,"interaction_only interaction_only: bool, default=FalseIf `True`, only interaction features are produced: features that areproducts of at most `degree` *distinct* input features, i.e. terms withpower of 2 or higher of the same input feature are excluded:- included: `x[0]`, `x[1]`, `x[0] * x[1]`, etc.- excluded: `x[0] ** 2`, `x[0] ** 2 * x[1]`, etc.",False
,"include_bias include_bias: bool, default=TrueIf `True` (default), then include a bias column, the feature in whichall polynomial powers are zero (i.e. a column of ones - acts as anintercept term in a linear model).",False
,"order order: {'C', 'F'}, default='C'Order of output array in the dense case. `'F'` order is faster tocompute, but may slow down subsequent estimators... versionadded:: 0.21",'C'
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06


In [11]:
poly_predictions = poly_model.predict(X_test)

In [12]:
poly_mae = mean_absolute_error(y_test, poly_predictions)
poly_mse = mean_squared_error(y_test, poly_predictions)
poly_rmse = np.sqrt(poly_mse)

print(f"Polynomial Regression MAE: {poly_mae:.3f}")
print(f"Polynomial Regression MSE: {poly_mse:.3f}")
print(f"Polynomial Regression RMSE: {poly_rmse:.3f}")

Polynomial Regression MAE: 2.702
Polynomial Regression MSE: 14.533
Polynomial Regression RMSE: 3.812


In [13]:
results = pd.DataFrame(
    {
        "MAE": [linear_mae, poly_mae],
        "MSE": [linear_mse, poly_mse],
        "RMSE": [linear_rmse, poly_rmse],
    },
    index=["Linear", "Polynomial(degree=2)"],
)
results

,MAE,MSE,RMSE
Linear,3.835696,28.547585,5.342994
Polynomial(degree=2),2.701704,14.533097,3.812230


### 3. Polynomial Regression Model
- Train a Polynomial Regression model
- Make predictions on the test set
- Compute evaluation metrics:
  - MAE
  - MSE
  - RMSE

In [14]:
better_model = "Linear" if linear_rmse <= poly_rmse else "Polynomial(degree=2)"
print(f"Better model on test data: {better_model}")

if better_model == "Linear":
    print(
        "Linear Regression is preferred because it has lower RMSE "
        "and is less likely to overfit for this dataset."
    )
else:
    print(
        "Polynomial Regression is preferred because it has lower RMSE "
        "and captures non-linear effects better."
    )

Better model on test data: Polynomial(degree=2)
Polynomial Regression is preferred because it has lower RMSE and captures non-linear effects better.


In [15]:
print("Test set R^2 scores:")
print(f"Linear R^2: {linear_model.score(X_test, y_test):.3f}")
print(f"Polynomial R^2: {poly_model.score(X_test, y_test):.3f}")

Test set R^2 scores:
Linear R^2: 0.712
Polynomial R^2: 0.854


In [16]:
print("Linear regression equation approximation:")
equation = "MEDV = " + f"{linear_model.intercept_:.3f}"
for name, coef in coefficients.items():
    equation += f" + ({coef:.3f} * {name})"
print(equation)

Linear regression equation approximation:
MEDV = 40.219 + (-0.089 * CRIM) + (0.050 * ZN) + (0.020 * INDUS) + (3.754 * CHAS) + (-17.748 * NOX) + (3.248 * RM) + (0.012 * AGE) + (-1.409 * DIS) + (0.264 * RAD) + (-0.010 * TAX) + (-0.952 * PTRATIO) + (0.006 * B) + (-0.597 * LSTAT)


In [17]:
example_row = X_test.iloc[[0]]
print("Example input row:")
display(example_row)
print("Actual MEDV:", float(y_test.iloc[0]))
print("Linear prediction:", float(linear_model.predict(example_row)[0]))
print("Polynomial prediction:", float(poly_model.predict(example_row)[0]))

Example input row:


,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT
195,0.01381,80.0,0.46,0.0,0.422,7.875,32.0,5.6484,4.0,255.0,14.4,394.23,2.97


Actual MEDV: 50.0
Linear prediction: 40.11113507518193
Polynomial prediction: 48.00472502159637


### 4. Model Comparison
- Compare Linear vs Polynomial Regression results
- Identify which model performs better on the test data and justify your decision

In [18]:
print("Comparison summary:")
print(results)
print("\nThe model with the lower RMSE on the test set is the best choice for this task.")

Comparison summary:
                           MAE        MSE      RMSE
Linear                3.835696  28.547585  5.342994
Polynomial(degree=2)  2.701704  14.533097  3.812230

The model with the lower RMSE on the test set is the best choice for this task.


In [19]:
if linear_rmse < poly_rmse:
    print("Linear Regression performs better on the test data.")
elif poly_rmse < linear_rmse:
    print("Polynomial Regression performs better on the test data.")
else:
    print("Both models perform equally on the test data.")

Polynomial Regression performs better on the test data.


In [20]:
print("Linear RMSE:", linear_rmse)
print("Polynomial RMSE:", poly_rmse)
print("Linear MAE:", linear_mae)
print("Polynomial MAE:", poly_mae)

Linear RMSE: 5.34299403625609
Polynomial RMSE: 3.8122299735995457
Linear MAE: 3.835696361418934
Polynomial MAE: 2.7017035453349805


In [21]:
print("Linear MSE:", linear_mse)
print("Polynomial MSE:", poly_mse)

Linear MSE: 28.547585271468137
Polynomial MSE: 14.533097371610793


In [22]:
print("Final recommendation:")
if linear_rmse <= poly_rmse:
    print("Choose Linear Regression due to simpler model and similar or better test performance.")
else:
    print("Choose Polynomial Regression because it gives better test accuracy.")


Final recommendation:
Choose Polynomial Regression because it gives better test accuracy.
